# Publication figures - Arrow-of-Time experiment

Static, **vector-PDF** figures for papers, posters, and slides - the
print-ready counterpart of the interactive `dashboard.html`. Eight
figures, mirroring the dashboard's main panels.

Each figure is a function in `analysis/figures.py`; this notebook drives
them and displays the result inline. PDFs are written under
`analysis/derived/figures/pub/` (gitignored - regeneratable; the figure
*code* is what's version-controlled).

Style: Helvetica throughout, seaborn `despine` with an offset for the
'torn-away' axes look.

**Prerequisite:** the analysis pipeline must have been run so the
derived TSVs/parquet exist (`analysis/derived/`). If they're missing or
stale:

```
python -m analysis.load_all
python -m analysis.per_subject
python -m analysis.per_video
python -m analysis.per_source
```


In [ ]:
# Make the `analysis` package importable from the notebook's folder.
import sys, pathlib
_root = pathlib.Path.cwd()
for _ in range(5):
    if (_root / 'analysis' / '__init__.py').is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("couldn't locate the analysis package - open the notebook from inside the repo")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

%matplotlib inline
from analysis import figures
from analysis.figures import PUB_DIR

PUB_DIR.mkdir(parents=True, exist_ok=True)
print('publication PDFs will be written to:', PUB_DIR)


## Figure 1 - Per-source arrow-of-time asymmetry

Forward-render vs backward-render identifiability, one point per source,
equal (1:1) axes.

- **Solid diagonal** `y = x` - raw symmetry. **Dashed diagonal**
  `y = x - offset` - bias-adjusted symmetry; the population forward
  bias shifts every clip off the solid line, so the dashed line is the
  real "no per-clip asymmetry" locus. The gap **is** the forward bias.
- **Colour** - `asymmetry_z_residual`: arctanh-decompressed,
  forward-bias removed. Orange = forward render carries the cleaner
  arrow-of-time cue; blue = the reversed render is the conspicuous one.
- **Size** - min views per direction. Labelled points = most extreme.


In [ ]:
fig1 = figures.per_source_asymmetry_figure(
    save_path=PUB_DIR / 'per_source_asymmetry.pdf',
    min_views=3, annotate_top=8,
)
fig1


## Figure 2 - Cohort quality

2x2 grid of distributions over the included Prolific cohort: type-1
sensitivity `d'`, metacognitive efficiency `M-ratio`, catch-direction
pass rate, and forward bias. Dashed red = cohort median; dotted dark =
a reference value (ideal M-ratio = 1, catch gate = 0.80, zero bias).


In [ ]:
fig2 = figures.cohort_quality_figure(
    save_path=PUB_DIR / 'cohort_quality.pdf',
    included_only=True,
)
fig2


## Figure 3 - Confidence calibration

Direction accuracy at each reported confidence level (1-5). One faint
line per subject; bold black = cohort mean with a shaded inter-quartile
band; dashed red = chance. A monotone rise = confidence tracks accuracy.


In [ ]:
fig3 = figures.calibration_figure(
    save_path=PUB_DIR / 'calibration.pdf',
    included_only=True,
)
fig3


## Figure 4 - Accuracy drift across main trials

Rolling-mean direction accuracy across the ordered real main trials.
One faint line per subject; bold black = cohort mean; dashed verticals =
main-block boundaries. A flat cohort mean = no fatigue / no warm-up.


In [ ]:
fig4 = figures.accuracy_drift_figure(
    save_path=PUB_DIR / 'accuracy_drift.pdf',
    included_only=True,
)
fig4


## Figure 5 - Per-video identifiability distribution

Distribution of the per-(clip x direction) identifiability score,
forward and backward overlaid. A leftward shift of the backward
distribution = reversed clips harder to identify (the Hanyu-et-al.
signature at the per-clip level).


In [ ]:
fig5 = figures.identifiability_figure(
    save_path=PUB_DIR / 'identifiability.pdf',
)
fig5


## Figure 6 - Confidence vs accuracy (the "fooler" view)

Mean confidence vs direction accuracy, one point per (clip x direction).
The shaded bottom-right quadrant - high confidence, low accuracy - holds
the "fooler" clips that visually suggest the opposite temporal
direction. The scientifically richest clips for arrow-of-time analysis.


In [ ]:
fig6 = figures.confidence_accuracy_figure(
    save_path=PUB_DIR / 'confidence_accuracy.pdf',
    min_views=3,
)
fig6


## Figure 7 - Corpus coverage

How many (stimulus x direction) cells have at least N viewings, against
the n = 20 per-cell sampling target (CLAUDE.md section 6).


In [ ]:
fig7 = figures.corpus_coverage_figure(
    save_path=PUB_DIR / 'corpus_coverage.pdf',
)
fig7


## Figure 8 - Inclusion gate

Methods figure: how many Prolific subjects pass the inclusion gate, and
which gate each excluded subject failed (a subject can fail more than
one - bar counts are per-reason).


In [ ]:
fig8 = figures.inclusion_figure(
    save_path=PUB_DIR / 'inclusion.pdf',
)
fig8


## Figure 9 - Identifiability-score stability (subsampling analysis)

A convergence check: as more participants watch a clip, does its
identifiability estimate stop moving? `analysis/stability.py` subsamples
the well-sampled cells (>= 20 views) and sweeps the viewer count k.

- **Left - precision.** m-out-of-n bootstrap SD of the score vs k.
  Lower = a k-viewer estimate wobbles less; the shaded band is the
  inter-quartile range across clips.
- **Right - ranking reliability.** Disjoint split-half correlation vs k
  (solid = directly measured), extended by the Spearman-Brown
  projection (dashed) past the measurable range; r -> 1 means the clip
  *ranking* has converged.

Forward and backward renders differ sharply: backward clips are noisier
per-clip but rank far more reliably (they span a much wider
identifiability range, so the ranking has more signal to work with).

The stability table is built on first use (a ~1 min subsampling run);
thereafter the figure reads `analysis/derived/stability.tsv`.


In [ ]:
fig9 = figures.stability_figure(
    save_path=PUB_DIR / 'stability.pdf',
)
fig9


## Regenerate everything / add a figure

`figures.build_all()` rebuilds **every** registered figure as a PDF in
one shot - handy after a fresh pipeline run:

```python
figures.build_all()
```

To add a new figure: write `my_figure(...)` in `analysis/figures.py`
(using the shared `_styled()` rcParams context and `_despine()` helper
for consistent typography and axes), register it in the `_FIGURES`
dict at the bottom of that file, and add a driving cell here.
